# Mapping the Parameter Space of $n_s$ and $r$

This notebook uses the `numerical_observables_calculation` pipeline to explore how initial field configurations $(x_i, y_i)$ affect the observed primordial power spectrum. 

We will:
1. Define a grid of initial conditions.
2. Run the exact numerical solver for each point.
3. Visualize the results using heatmaps (how $n_s$ varies) and trajectory plots in the $n_s$ vs $r$ plane.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 16, 'axes.labelsize': 20, 'legend.fontsize': 16, 'xtick.labelsize': 16, 'ytick.labelsize': 16})
import pandas as pd
import seaborn as sns
import sys
import os
from tqdm.notebook import tqdm
import json
import uuid
import datetime
from scipy.interpolate import make_interp_spline
import ipywidgets as widgets
from IPython.display import display, clear_output


# Import our new pipeline and model
sys.path.append(os.path.abspath('..'))
from models import HiggsModel
import numerical_observables_calculation as calc

# Setup Plotting
plt.style.use('seaborn-v0_8-paper')
sns.set_context("talk")
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'medium',
         'ytick.labelsize':'medium'}
plt.rcParams.update(params)
sns.set_palette("viridis")

## 1. Grid Definition
We explore a range of initial values around the standard USR (Ultra Slow Roll) transition points for Higgs Inflation.

In [40]:
model = HiggsModel(xi=15000, lam=0.13)

region_left  = np.linspace(5.4, 5.6, 80, endpoint=False) 
region_break = np.linspace(5.6, 5.7, 120, endpoint=False) 
region_right = np.linspace(5.7,5.8, 60)

grid_res = 100
yi_vals = np.linspace(-0.1, -0.1, 1)
phi0_vals = np.concatenate([region_left, region_break, region_right])

all_results = []

In [41]:
print(f"Starting grid search on {len(phi0_vals) * len(yi_vals)} configurations...")

save_bol = False #Save each and every run data into JSON? Be careful will create a lot of cluter. 

for p in tqdm(phi0_vals, desc="Phi0 Index"):
    for v in yi_vals:
        custom_T_span = np.linspace(0, 1000, 50000)
        res = calc.run_inflation_protocol(model, phi0=p, yi=v, delta=1e-5, T_span_bg=custom_T_span, save_to_file=save_bol)

        
        if res["status"] == "success":
            all_results.append({
                'phi0': p,
                'yi': v,
                'ns': res['ns'],
                'r': res['r'],
                'ns_SR': res['ns_SR'],
                'r_SR': res['r_SR'],
                'N_total': res['N_total'],
                'Ps': res['P_S']
            })


df = pd.DataFrame(all_results)
print("Grid search complete!")


Starting grid search on 260 configurations...


Phi0 Index:   0%|          | 0/260 [00:00<?, ?it/s]

Grid search complete!


In [44]:


# 1. Extract bounds for the metadata and filename
phi0_min = min(phi0_vals)
phi0_max = max(phi0_vals)
yi_min = min(yi_vals)
yi_max = max(yi_vals)



# 2. Build the structured JSON dictionary dynamically
grid_output_data = {
    "metadata": {
        "run_id": uuid.uuid4().hex[:8],
        "timestamp": datetime.datetime.now().isoformat(),
        "description": "Grid search mapping ns and r sensitivity to initial conditions."
    },
    "model_parameters": {
        "name": "Higgs Inflation",
        "xi": model.xi_val,
        "lambda": model.lam
    },
    "grid_parameters": {
        "phi0_min": phi0_min,
        "phi0_max": phi0_max,
        "phi0_steps": len(phi0_vals),
        "yi_min": yi_min,
        "yi_max": yi_max,
        "yi_steps": len(yi_vals),
        "total_configurations_attempted": len(phi0_vals) * len(yi_vals),
        "successful_simulations": len(all_results)
    },
    "numerical_settings": {
        "delta_finite_difference": 1e-05,
        "T_span_bg_max": 1000,
        "T_span_bg_steps": 10000
    },
    "results": all_results
}



# 3. Create the save path and directory
save_dir = '../outputs'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 4. Generate a descriptive filename with bounds and run_id
filename = f"grid_search_phi0_{phi0_min:.2f}to{phi0_max:.2f}_yi_{abs(yi_min):.2f}to{abs(yi_max):.2f}_{grid_output_data['metadata']['run_id']}.json"
filepath = os.path.join(save_dir, filename)

# 5. Write the data to a JSON file
with open(filepath, 'w') as f:
    json.dump(grid_output_data, f, indent=4)

print(f"✅ Grid search results securely saved to: {filepath}")


✅ Grid search results securely saved to: ../outputs\grid_search_phi0_5.40to5.80_yi_0.10to0.10_c42e4760.json


## 2. Visualizing how $n_s$ moves in the Initial Condition Plane

We use heatmaps to see where the field configuration leads to higher or lower spectral indices.

In [ ]:


# Add scripts directory to path and import modular functions
sys.path.append(os.path.abspath('../scripts'))
import analysis_utils as utils

# --- Initialize Variables ---
sim_history = utils.scan_simulations()
output_area = widgets.Output()

# --- Define UI Elements ---
file_select = widgets.Dropdown(
    options=[(r['label'], r['path']) for _, r in sim_history.iterrows()],
    description='1. Select Run:', style={'description_width': 'initial'}, layout={'width': '450px'}
)

yi_select = widgets.Dropdown(description='2. Select yi:', style={'description_width': 'initial'})

def update_dashboard(*args):
    global loaded_data, df, df_slice, current_yi # Variables available notebook-wide
    
    with output_area:
        clear_output(wait=True)
        # 1. Load File
        with open(file_select.value, 'r') as f:
            loaded_data = json.load(f)
        df = pd.DataFrame(loaded_data["results"])
        
        # 2. Update yi choices based on this file if not already set
        available_yi = sorted(df['yi'].unique())
        yi_select.options = [(f"{y:.3f}", y) for y in available_yi]
        
        # 3. Get Slice and Plot
        df_slice, current_yi = utils.get_yi_slice(df, target_yi=yi_select.value)
        
        print(f"✅ Run: {loaded_data['metadata']['run_id']} | File: {os.path.basename(file_select.value)}")
        utils.plot_ms_vs_sr_comparison(df_slice, current_yi)

# --- Link everything ---
file_select.observe(update_dashboard, names='value')
yi_select.observe(update_dashboard, names='value')

display(widgets.VBox([file_select, yi_select]), output_area)

# Initial trigger
update_dashboard()


Output()